<a href="https://colab.research.google.com/github/dhruvjoshi137/Data_Structures_Algorithms/blob/main/NEW_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set model save path in Google Drive
save_directory = "/content/drive/MyDrive/my_emotion_model"

In [ ]:
# Install dependencies
!pip install transformers datasets evaluate fsspec==2023.6.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-runtime-cu12==12.4.127; platform_system == "Linux" and platform_mac

In [ ]:
# Load Dataset and Tokenizer
from datasets import load_dataset
from transformers import AutoTokenizer
model_checkpoint = "roberta-large"
dataset = load_dataset("go_emotions", "simplified")
train_dataset = dataset["train"]
val_dataset = dataset["validation"]
test_dataset = dataset["test"]

label_names = train_dataset.features["labels"].feature.names
num_labels = len(label_names)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
# Tokenize and prepare labels
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

def prepare_labels_binary(examples):
    batch_labels = []
    for label_list in examples["labels"]:
        one_hot = [0.0] * num_labels
        for idx in label_list:
            one_hot[idx] = 1.0
        batch_labels.append(one_hot)
    examples["labels"] = batch_labels
    return examples

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.map(prepare_labels_binary, batched=True)
val_dataset = val_dataset.map(prepare_labels_binary, batched=True)
test_dataset = test_dataset.map(prepare_labels_binary, batched=True)

train_dataset = train_dataset.remove_columns(["text", "id"])
val_dataset = val_dataset.remove_columns(["text", "id"])
test_dataset = test_dataset.remove_columns(["text", "id"])

import torch

def enforce_float_transform(example):
    example["labels"] = torch.tensor(example["labels"], dtype=torch.float32)
    return example

train_dataset = train_dataset.with_transform(enforce_float_transform)
val_dataset = val_dataset.with_transform(enforce_float_transform)
test_dataset = test_dataset.with_transform(enforce_float_transform)


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [ ]:
# Compute positive class weights
import numpy as np
label_matrix = np.zeros((len(dataset["train"]), num_labels))
for i, labels in enumerate(dataset["train"]["labels"]):
    for label in labels:
        label_matrix[i][label] = 1
label_counts = label_matrix.sum(axis=0)
total = len(dataset["train"])
pos_weights = (total - label_counts) / label_counts

In [ ]:
# Define custom model
from transformers import AutoModel
import torch.nn as nn

class CustomRobertaModel(nn.Module):
    def __init__(self, checkpoint, num_labels, pos_weights):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weights, dtype=torch.float))

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)
        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

model = CustomRobertaModel(model_checkpoint, num_labels, pos_weights)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Training arguments
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./results_goemotions",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="exact_match_accuracy",
    greater_is_better=True,
    report_to="none",
    gradient_accumulation_steps=2,
    fp16=True
)

In [ ]:
# Data collator
from transformers import DataCollatorWithPadding
base_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def custom_collator(features):
    labels = [f["labels"] for f in features]
    for f in features:
        del f["labels"]
    batch = base_collator(features)
    batch["labels"] = torch.stack(labels)
    return batch

In [ ]:
# Compute metrics
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.from_numpy(logits)).numpy()
    thresholds = np.arange(0.3, 0.6, 0.05)
    best_thresh = 0.5
    best_ema = 0

    for t in thresholds:
        preds = (probs >= t).astype(int)
        ema = np.mean(np.all(preds == labels, axis=1))
        if ema > best_ema:
            best_ema = ema
            best_thresh = t

    final_preds = (probs >= best_thresh).astype(int)
    macro_f1 = f1_score(labels, final_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(labels, final_preds, average='micro', zero_division=0)
    exact_match = np.mean(np.all(final_preds == labels, axis=1))

    top_k_hits = 0
    for i in range(len(labels)):
        top3_pred = probs[i].argsort()[-3:]
        true_indices = np.where(labels[i] == 1)[0]
        if any(label in top3_pred for label in true_indices):
            top_k_hits += 1
    top3_accuracy = top_k_hits / len(labels)

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "exact_match_accuracy": exact_match,
        "top_3_accuracy": top3_accuracy
    }


In [ ]:
# Trainer
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=custom_collator,
    compute_metrics=compute_metrics
)

print("\nStarting training...")
trainer.train()

print("\nEvaluating on test set...")
results = trainer.evaluate(test_dataset)
print(results)

/tmp/ipython-input-12-3980620642.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Starting training...


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,Exact Match Accuracy,Top 3 Accuracy
1,0.695300,0.598740,0.371077,0.396877,0.075746,0.772208
2,0.579100,0.598419,0.393096,0.418050,0.092518,0.789716


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,Exact Match Accuracy,Top 3 Accuracy
1,0.695300,0.598740,0.371077,0.396877,0.075746,0.772208
2,0.579100,0.598419,0.393096,0.418050,0.092518,0.789716
3,0.449800,0.604499,0.403570,0.442225,0.115370,0.806672
4,0.357100,0.653510,0.435519,0.479608,0.162366,0.833395
5,0.280200,0.698865,0.445192,0.492729,0.178032,0.844453
6,0.236600,0.770925,0.466075,0.511823,0.214154,0.850903



Evaluating on test set...


{'eval_loss': 0.7839110493659973, 'eval_macro_f1': 0.4579024774091905, 'eval_micro_f1': 0.507206733298264, 'eval_exact_match_accuracy': 0.2017689331122167, 'eval_top_3_accuracy': 0.8468767274737424, 'eval_runtime': 8.8952, 'eval_samples_per_second': 610.102, 'eval_steps_per_second': 19.111, 'epoch': 6.0}


In [ ]:
# Save to Google Drive manually
torch.save(model.state_dict(), f"{save_directory}/pytorch_model.bin")
tokenizer.save_pretrained(save_directory)


('/content/drive/MyDrive/my_emotion_model/tokenizer_config.json',
 '/content/drive/MyDrive/my_emotion_model/special_tokens_map.json',
 '/content/drive/MyDrive/my_emotion_model/vocab.json',
 '/content/drive/MyDrive/my_emotion_model/merges.txt',
 '/content/drive/MyDrive/my_emotion_model/added_tokens.json',
 '/content/drive/MyDrive/my_emotion_model/tokenizer.json')

In [ ]:
# Load manually for inference
model = CustomRobertaModel(model_checkpoint, num_labels, pos_weights)
model.load_state_dict(torch.load(f"{save_directory}/pytorch_model.bin", map_location=torch.device('cpu')))
model.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CustomRobertaModel(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
           

In [ ]:
# Prediction functions
label_names = ["admiration", "amusement", "anger", "annoyance", "approval",
               "caring", "confusion", "curiosity", "desire", "disappointment",
               "disapproval", "disgust", "embarrassment", "excitement", "fear",
               "gratitude", "grief", "joy", "love", "nervousness", "optimism",
               "pride", "realization", "relief", "remorse", "sadness", "surprise",
               "neutral"]

def predict_emotions(text, model, tokenizer, label_names, threshold=0.5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        probs = torch.sigmoid(logits)

    probs = probs.cpu().numpy()[0]
    predicted_labels = [label_names[i] for i, prob in enumerate(probs) if prob >= threshold]
    return predicted_labels

def predict_top3_emotions(text, model, tokenizer, label_names):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        probs = torch.sigmoid(logits)

    probs = probs.cpu().numpy()[0]
    top3_indices = probs.argsort()[-3:][::-1]
    top3_labels = [(label_names[i], round(float(probs[i]), 3)) for i in top3_indices]

    return top3_labels

In [ ]:
# CLI testing
print("Ready for emotion predictions! Type your text below.\nType 'exit' to stop.\n")
while True:
    user_input = input("Enter a sentence: ")
    if user_input.lower() == "exit":
        break
    try:
        predicted_labels = predict_emotions(user_input, model, tokenizer, label_names)
        top3 = predict_top3_emotions(user_input, model, tokenizer, label_names)

        print("Predicted Emotions:", predicted_labels)
        print("Top 3 Emotions (with confidence):", top3)
    except Exception as e:
        print("Error:", e)

Ready for emotion predictions! Type your text below.
Type 'exit' to stop.

Enter a sentence: I finally got the job I’ve always dreamed of!
Predicted Emotions: ['approval', 'desire', 'excitement', 'joy', 'pride', 'realization', 'relief']
Top 3 Emotions (with confidence): [('pride', 0.988), ('excitement', 0.978), ('joy', 0.961)]
Enter a sentence: Today was so peaceful and calm. I really needed that.
Predicted Emotions: ['approval', 'desire', 'gratitude', 'joy', 'realization', 'relief']
Top 3 Emotions (with confidence): [('relief', 0.977), ('desire', 0.737), ('realization', 0.697)]
Enter a sentence: I feel completely overwhelmed and hopeless right now.
Predicted Emotions: ['annoyance', 'disappointment', 'sadness']
Top 3 Emotions (with confidence): [('sadness', 0.986), ('disappointment', 0.957), ('annoyance', 0.577)]
Enter a sentence: I miss the old days, but I’m happy where I am now.
Predicted Emotions: ['disappointment', 'joy', 'sadness']
Top 3 Emotions (with confidence): [('joy', 0.998)

In [ ]:
import os

model_path = "/content/drive/MyDrive/my_emotion_model"
files = os.listdir(model_path)
print("Files in saved model directory:", files)


Files in saved model directory: ['model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin', 'pytorch_model.bin']


In [ ]:
model_code = """
import torch
import torch.nn as nn
from transformers import AutoModel

class CustomRobertaModel(nn.Module):
    def __init__(self, checkpoint, num_labels, pos_weights):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weights, dtype=torch.float))

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)
        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}
"""

with open("/content/drive/MyDrive/my_emotion_model/model_architecture.py", "w") as f:
    f.write(model_code)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/my_emotion_model/model_architecture.py'